In [10]:
import os
from flask import Flask, render_template, send_from_directory
from core.persistence.database_manager import DatabaseManager
db_manager = DatabaseManager("Compte") 
compta= db_manager.comptabilite

In [11]:
import pandas as pd

def clean_mixed_dates(df, col='Date'):
    def parse_date(x):
        if pd.isna(x) or x == 'None':
            return pd.NaT
        x = str(x).strip()
        try:
            # On tente le format ISO (2026-03-28)
            if '-' in x:
                return pd.to_datetime(x, format='%Y-%m-%d')
            # On tente le format FR (01/04/2026)
            elif '/' in x:
                return pd.to_datetime(x, format='%d/%m/%Y')
        except:
            # Si échec, on laisse Pandas deviner de façon flexible
            return pd.to_datetime(x, errors='coerce')
        return pd.NaT

    df[col] = df[col].apply(parse_date)
    
    # Tri chronologique pour vérifier la cohérence
    df = df.sort_values(by=col, ascending=False).reset_index(drop=True)
    return df




In [14]:
for compte in  db_manager.comptabilite.liste_compte :
    #print(compte.df.head(10))
    compte.df = clean_mixed_dates(compte.df)
    print(compte.df.head(10))
    compte.triage("Date",False)
    lignes_erreurs = compte.df[compte.df['Date'].isna()]

    print("--- Lignes posant problème ---")
    print(lignes_erreurs[['Intitule', 'Valeur','Date']]) 

    

        Date                  Intitule  Categorie          Classe     Type  \
0 2026-12-04     Erreur lié a l'app v4     Erreur          Erreur  Depense   
1 2026-04-16       Assurance swisscare  Essentiel       Assurance  Depense   
2 2026-04-16                   SBB CHF     Voyage             SBB  Depense   
3 2026-04-14                    course  Essentiel      Nourriture  Depense   
4 2026-04-07                  virement   Virement  Compte_Courant  Depense   
5 2026-04-03           Plein d essence     Voyage         voiture  Depense   
6 2026-04-03               burger King     Loisir      Nourriture  Depense   
7 2026-04-01                     Loyer  Essentiel        Logement  Depense   
8 2026-03-31                    course  Essentiel      Nourriture  Depense   
9 2026-03-28  Framboisier + Blue berry     Loisir          Plante  Depense   

    Valeur  
0   267.00  
1    38.00  
2    62.00  
3    62.68  
4  2000.00  
5    59.38  
6    22.72  
7  1400.00  
8   179.97  
9    51.10 